In [ ]:
import os
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Detect hardware
try:
    tpu = None
    strategy = tf.distribute.MirroredStrategy()
    print(f'Running on {strategy.num_replicas_in_sync} GPUs')
except Exception as e:
    strategy = tf.distribute.get_strategy()
    print('Running on single CPU/GPU')

# Memory Management: Clear backend to start fresh
tf.keras.backend.clear_session()

: 

In [2]:
# SET THESE MANUALLY based on your GPU and model choice
CONFIG = {
    "img_height": 224,        # Standard for ResNet50
    "img_width": 224,
    "channels": 3,
    "batch_size_per_replica": 32, # Increase if you have more than 15GB VRAM per GPU
    "epochs": 10,             # Number of training loops
    "learning_rate": 0.001,   # Use 0.001 for dummy, 1e-5 for fine-tuning
    "main_classes": 0,        # Leave as 0; Cell 3 will update this
    "sub_classes": 0,         # Leave as 0; Cell 3 will update this
}

In [ ]:
import os
#os.chdir('/kaggle/input/datasets/obulisainaren/multi-cancer/Multi Cancer/Multi Cancer')
os.listdir('/kaggle')

In [4]:
import os
import glob
import tensorflow as tf
from sklearn.model_selection import train_test_split

# --- 1. ENSURE GLOBAL VARIABLES ARE SET ---
try:
    GLOBAL_BATCH_SIZE = CONFIG["batch_size_per_replica"] * strategy.num_replicas_in_sync
except NameError:
    GLOBAL_BATCH_SIZE = 64 
    print("⚠️ Warning: CONFIG or strategy not found. Using default Batch Size: 64")

# --- 2. SET THE DATASET PATH ---
DATASET_PATH = "/kaggle/input/datasets/obulisainaren/multi-cancer/Multi Cancer/Multi Cancer" 

# --- 3. DYNAMICALLY CALCULATE CLASSES ---
main_class_names = sorted([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))])

all_sub_paths = glob.glob(os.path.join(DATASET_PATH, "*/*"))
sub_class_names = sorted(list(set([os.path.basename(p) for p in all_sub_paths if os.path.isdir(p)])))

CONFIG["main_classes"] = len(main_class_names)
CONFIG["sub_classes"] = len(sub_class_names)

main_map = {name: i for i, name in enumerate(main_class_names)}
sub_map = {name: i for i, name in enumerate(sub_class_names)}

print(f"✅ Found {CONFIG['main_classes']} Main Classes and {CONFIG['sub_classes']} Sub-Classes.")

# --- 4. GATHER ALL IMAGE PATHS AND LABELS ---
all_images = []
main_labels_int = []
sub_labels_int = []

for main_cat in main_class_names:
    main_path = os.path.join(DATASET_PATH, main_cat)
    sub_cats = [d for d in os.listdir(main_path) if os.path.isdir(os.path.join(main_path, d))]
    
    for sub_cat in sub_cats:
        folder_path = os.path.join(main_path, sub_cat)
        images_in_folder = glob.glob(os.path.join(folder_path, "*.*"))
        
        for img_path in images_in_folder:
            all_images.append(img_path)
            main_labels_int.append(main_map[main_cat])
            sub_labels_int.append(sub_map[sub_cat])

print(f"✅ Total images found: {len(all_images)}")

# --- 5. SPLIT DATA (80% Train, 20% Val) ---
train_paths, val_paths, train_main, val_main, train_sub, val_sub = train_test_split(
    all_images, main_labels_int, sub_labels_int, test_size=0.2, random_state=42
)

# --- 6. TF.DATA PIPELINE FUNCTIONS ---
def process_real_image(path, main_label, sub_label):
    img = tf.io.read_file(path)
    
    is_jpg = tf.strings.regex_full_match(path, r".*\.jpg")
    
    img = tf.cond(is_jpg, 
                  lambda: tf.image.decode_jpeg(img, channels=3), 
                  lambda: tf.image.decode_png(img, channels=3))
    
    img = tf.image.resize(img, [CONFIG["img_height"], CONFIG["img_width"]])
    img = tf.keras.applications.resnet50.preprocess_input(img)
    
    return img, {"main_output": main_label, "sub_output": sub_label}

def create_real_dataset(paths, main_labels, sub_labels, is_training=True):
    dataset = tf.data.Dataset.from_tensor_slices((paths, main_labels, sub_labels))
    
    # FIX: Completed the line below with tf.data.AUTOTUNE)
    dataset = dataset.map(process_real_image, num_parallel_calls=tf.data.AUTOTUNE)
    
    # FIX: Added the missing batching and prefetching logic that got cut off
    if is_training:
        dataset = dataset.shuffle(buffer_size=1024)
    
    dataset = dataset.batch(GLOBAL_BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# --- 7. INITIALIZE DATASETS ---
train_ds = create_real_dataset(train_paths, train_main, train_sub, is_training=True)
val_ds = create_real_dataset(val_paths, val_main, val_sub, is_training=False)

print("Datasets are ready for training.")

✅ Found 8 Main Classes and 26 Sub-Classes.
✅ Total images found: 130002
Datasets are ready for training.


In [5]:
with strategy.scope():
    # Base ResNet50
    base_model = tf.keras.applications.ResNet50(
        include_top=False, 
        weights='imagenet', 
        input_shape=(CONFIG["img_height"], CONFIG["img_width"], 3)
    )
    base_model.trainable = False # Freeze for initial dummy training

    inputs = tf.keras.Input(shape=(CONFIG["img_height"], CONFIG["img_width"], 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    # --- Head 1: Main Class ---
    main_branch = tf.keras.layers.Dense(256, activation='relu')(x)
    main_output = tf.keras.layers.Dense(
        CONFIG["main_classes"], 
        activation='softmax', 
        name='main_output'
    )(main_branch)

    # --- Head 2: Sub Class (Concatenated with Main Class features) ---
    sub_branch = tf.keras.layers.Concatenate()([x, main_branch])
    sub_branch = tf.keras.layers.Dense(256, activation='relu')(sub_branch)
    sub_output = tf.keras.layers.Dense(
        CONFIG["sub_classes"], 
        activation='softmax', 
        name='sub_output'
    )(sub_branch)

    model = tf.keras.Model(inputs=inputs, outputs=[main_output, sub_output])

    # Compilation
    # ... (rest of the model architecture code)

    # Updated Compilation Logic
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"]),
        loss={
            "main_output": "sparse_categorical_crossentropy",
            "sub_output": "sparse_categorical_crossentropy"
        },
        loss_weights={
            "main_output": 0.4, 
            "sub_output": 1.0
        },
        # Map metrics explicitly to each output name
        metrics={
            "main_output": "accuracy",
            "sub_output": "accuracy"
        }
    )

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ input_layer_1[0]… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,049,088 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 768)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │    196,864 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ main_output (Dense) │ (None, 8)         │      2,056 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sub_output (Dense)  │ (None, 26)        │      6,682 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,973,730 (95.27 MB)

 Trainable params: 1,386,018 (5.29 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
print("Starting Training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    verbose=1
)

Starting Training...
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

I0000 00:00:1775456095.603770     141 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775456095.603766     140 cuda_dnn.cc:529] Loaded cuDNN version 91002


 176/1626 ━━━━━━━━━━━━━━━━━━━━ 10:02 415ms/step - loss: 1.1840 - main_output_accuracy: 0.9126 - main_output_loss: 0.2728 - sub_output_accuracy: 0.6407 - sub_output_loss: 1.0749

In [ ]:
# Save weights for the real training script later

# Visualization
acc_main = history.history['main_output_accuracy']
acc_sub = history.history['sub_output_accuracy']
epochs_range = range(CONFIG["epochs"])

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc_main, label='Main Class Accuracy')
plt.plot(epochs_range, acc_sub, label='Sub Class Accuracy')
plt.title('Training Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Total Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. REVERSE MAPPINGS (ID to String) ---
# We need these to turn the integer predictions back into readable text
rev_main_map = {v: k for k, v in main_map.items()}
rev_sub_map = {v: k for k, v in sub_map.items()}

# --- 2. OVERALL EVALUATION ---
print("📊 Evaluating model on the validation dataset...")
results = model.evaluate(val_ds, verbose=1)

print("\n--- Final Validation Metrics ---")
# Dynamically print whatever metrics the model compiled with
for name, value in zip(model.metrics_names, results):
    print(f"{name.upper()}: {value:.4f}")

# --- 3. VISUAL PREDICTION TEST ---
print("\n🖼️ Generating visual predictions for a sample batch...")

# Take exactly one batch from the validation dataset
for images, labels in val_ds.take(1):
    
    # Extract the true labels from the dictionary
    true_main = labels['main_output'].numpy()
    true_sub = labels['sub_output'].numpy()
    
    # Run the model to get predictions
    # Note: Because the model has 2 heads, preds is a list of two arrays
    preds = model.predict(images)
    pred_main = np.argmax(preds[0], axis=1) # index 0 is main_output
    pred_sub = np.argmax(preds[1], axis=1)  # index 1 is sub_output
    
    # Set up a 3x3 plot
    plt.figure(figsize=(12, 12))
    
    # Loop through the first 9 images in the batch
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        
        # De-process the image for displaying 
        # (ResNet50 preprocessing shifts colors, so we normalize back to 0-1 for matplotlib)
        img = images[i].numpy()
        img = ((img - img.min()) / (img.max() - img.min()))
        
        plt.imshow(img)
        
        # Build the label text
        actual_text = f"TRUE: {rev_main_map[true_main[i]]} \n({rev_sub_map[true_sub[i]]})"
        pred_text = f"PRED: {rev_main_map[pred_main[i]]} \n({rev_sub_map[pred_sub[i]]})"
        
        # Color the text Green if 100% correct, Red if any part is wrong
        is_correct = (true_main[i] == pred_main[i]) and (true_sub[i] == pred_sub[i])
        title_color = "darkgreen" if is_correct else "darkred"
        
        plt.title(actual_text + "\n---\n" + pred_text, color=title_color, fontsize=10, fontweight='bold')
        plt.axis("off")
        
    plt.tight_layout()
    plt.show()
    
    # Break out of the loop so we only do one batch
    break

: 

In [ ]:
import gc
with strategy.scope():gc.collect()

In [8]:
from tensorflow.keras.models import load_model

with strategy.scope():

    model = load_model('/kaggle/working/resnet50_mod2.keras')
    base_model = model.layers[1] 
    base_model.trainable = True
    for i in base_model.layers[:-15]:
        i.trainable = False
    
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6), # Use a tiny learning rate!
        loss={
            "main_output": "sparse_categorical_crossentropy",
            "sub_output": "sparse_categorical_crossentropy"
        },
        metrics={"main_output": "accuracy", "sub_output": "accuracy"}
    )
    
    # Now you are ready to call model.fit() on your real data
model.summary()
base_model.summary()
sum(1 for i in base_model.layers if i.trainable)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ input_layer_1[0]… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,049,088 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 768)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │    196,864 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ main_output (Dense) │ (None, 8)         │      2,056 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sub_output (Dense)  │ (None, 26)        │      6,682 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,973,730 (95.27 MB)

 Trainable params: 6,906,402 (26.35 MB)

 Non-trainable params: 18,067,328 (68.92 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 5,520,384 (21.06 MB)

 Non-trainable params: 18,067,328 (68.92 MB)

15